# Gerar CSV Censo Escolar

Notebook simplificado para gerar `trabalho/censo-escolar.csv` com leitura e escrita em lotes para baixo uso de memória.

## 1) Configuração

In [1]:
from pathlib import Path
import csv
import re

import pandas as pd

RAIZ_PROJETO = Path.cwd()
if not (RAIZ_PROJETO / 'datasets').exists() and (RAIZ_PROJETO.parent / 'datasets').exists():
    RAIZ_PROJETO = RAIZ_PROJETO.parent

DIR_DADOS = RAIZ_PROJETO / 'datasets'
DIR_TRABALHO = RAIZ_PROJETO / 'trabalho'
CAMINHO_SAIDA = DIR_TRABALHO / 'censo-escolar.csv'

ANOS = range(1995, 2026)
TAMANHO_CHUNK = 50_000
MAX_COLUNAS_SAIDA = 100
MAX_COLUNAS_FONTE = 300

DIR_TRABALHO.mkdir(parents=True, exist_ok=True)

print(f'Bases de dados: {DIR_DADOS}')
print(f'Saída: {CAMINHO_SAIDA}')


Bases de dados: /home/gabriel/ciencias-computacao/programacao-para-analise-de-dados/datasets
Saída: /home/gabriel/ciencias-computacao/programacao-para-analise-de-dados/trabalho/censo-escolar.csv


### 2) Especificação das colunas

In [2]:
def especificacao(tipo, apelidos):
    return {'tipo': tipo, 'apelidos': apelidos}

ESPECIFICACOES_COLUNAS = {
    'ano_censo': especificacao('inteiro', ['NU_ANO_CENSO', 'ANO', 'NU_ANO']),
    'id_escola': especificacao('texto', ['CO_ENTIDADE', 'MASCARA']),
    'co_municipio': especificacao('texto', ['CO_MUNICIPIO', 'CODMUNIC', 'CO_IBGE']),
    'no_municipio': especificacao('texto', ['NO_MUNICIPIO', 'MUNIC']),
    'sg_uf': especificacao('texto', ['SG_UF', 'SIGLA']),
    'co_uf': especificacao('inteiro', ['CO_UF']),
    'no_uf': especificacao('texto', ['NO_UF', 'UF']),
    'no_regiao': especificacao('texto', ['NO_REGIAO']),
    'co_regiao': especificacao('inteiro', ['CO_REGIAO']),
    'no_entidade': especificacao('texto', ['NO_ENTIDADE']),
    'dependencia_administrativa': especificacao('categoria_dependencia', ['TP_DEPENDENCIA', 'DEP']),
    'localizacao': especificacao('categoria_localizacao', ['TP_LOCALIZACAO', 'LOC']),
    'situacao_funcionamento': especificacao('categoria_situacao', ['TP_SITUACAO_FUNCIONAMENTO', 'CODFUNC']),
    'tp_localizacao_diferenciada': especificacao('inteiro', ['TP_LOCALIZACAO_DIFERENCIADA']),
    'tp_categoria_escola_privada': especificacao('inteiro', ['TP_CATEGORIA_ESCOLA_PRIVADA']),

    'in_regular': especificacao('indicador', ['IN_REGULAR', 'ENSREGULAR', 'NIVELCRE', 'NIVELPRE', 'NIV_1GRAU', 'NIV_2GRAU', 'NIV_F1A4_8', 'NIV_F5A8_8', 'NIVELMED']),
    'in_creche': especificacao('indicador', ['IN_INF_CRE', 'IN_COMUM_CRECHE', 'IN_ESP_EXCLUSIVA_CRECHE', 'NIVELCRE']),
    'in_pre_escola': especificacao('indicador', ['IN_INF_PRE', 'IN_COMUM_PRE', 'IN_ESP_EXCLUSIVA_PRE', 'NIVELPRE']),
    'in_fundamental': especificacao('indicador', ['IN_FUND', 'IN_FUND_AI', 'IN_FUND_AF', 'IN_COMUM_FUND_AI', 'IN_COMUM_FUND_AF', 'NIV_1GRAU', 'NIV_F1A4_8', 'NIV_F5A8_8', 'NIV_F1A4', 'NIV_F5A8']),
    'in_medio': especificacao('indicador', ['IN_MED', 'IN_COMUM_MEDIO_MEDIO', 'IN_COMUM_MEDIO_INTEGRADO', 'NIV_2GRAU', 'NIVELMED', 'NIVELMEDIO']),
    'in_eja': especificacao('indicador', ['IN_EJA', 'IN_COMUM_EJA_FUND', 'IN_COMUM_EJA_MEDIO', 'ENSSUPLET', 'SUPL_AVA', 'SUPL_SAVA']),
    'in_profissionalizante': especificacao('indicador', ['IN_PROFISSIONALIZANTE', 'IN_COMUM_PROF', 'IN_PROF', 'IN_PROF_TEC', 'EDPROFIS']),
    'in_especial': especificacao('indicador', ['IN_ESPECIAL_EXCLUSIVA', 'IN_ESP', 'IN_ESP_CC', 'IN_ESP_CE', 'ESP_EXCL', 'ESP_T_ES', 'ENS_INCL']),
    'in_educacao_indigena': especificacao('indicador', ['IN_EDUCACAO_INDIGENA', 'ED_INDIG']),

    'in_predio_escolar': especificacao('indicador', ['IN_LOCAL_FUNC_PREDIO_ESCOLAR', 'PRED_ESC']),
    'in_predio_compartilhado': especificacao('indicador', ['IN_PREDIO_COMPARTILHADO', 'PRED_COM']),
    'in_agua_potavel': especificacao('indicador', ['IN_AGUA_POTAVEL', 'AGUA_FIL']),
    'in_agua_rede_publica': especificacao('indicador', ['IN_AGUA_REDE_PUBLICA', 'AGUA_PUB']),
    'in_agua_poco_artesiano': especificacao('indicador', ['IN_AGUA_POCO_ARTESIANO', 'AGUA_ART']),
    'in_agua_inexistente': especificacao('indicador', ['IN_AGUA_INEXISTENTE', 'AGUA_INE']),
    'in_energia_rede_publica': especificacao('indicador', ['IN_ENERGIA_REDE_PUBLICA', 'ENER_PUB']),
    'in_energia_gerador': especificacao('indicador', ['IN_ENERGIA_GERADOR_FOSSIL', 'IN_ENERGIA_GERADOR', 'ENER_GER']),
    'in_energia_inexistente': especificacao('indicador', ['IN_ENERGIA_INEXISTENTE', 'ENER_INE']),

    'qt_mat_bas': especificacao('inteiro', ['QT_MAT_BAS']),
    'qt_mat_fund': especificacao('inteiro', ['QT_MAT_FUND']),
    'qt_mat_med': especificacao('inteiro', ['QT_MAT_MED']),
    'qt_doc_bas': especificacao('inteiro', ['QT_DOC_BAS']),
    'qt_doc_fund': especificacao('inteiro', ['QT_DOC_FUND']),
    'qt_doc_med': especificacao('inteiro', ['QT_DOC_MED']),
    'qt_tur_bas': especificacao('inteiro', ['QT_TUR_BAS']),
    'qt_tur_fund': especificacao('inteiro', ['QT_TUR_FUND']),
    'qt_tur_med': especificacao('inteiro', ['QT_TUR_MED']),
}

COLUNAS_SAIDA = list(ESPECIFICACOES_COLUNAS)[:MAX_COLUNAS_SAIDA]
print(f'Colunas finais: {len(COLUNAS_SAIDA)}')


Colunas finais: 42


### 3) Descoberta dos arquivos

In [3]:
def extrair_ano(caminho):
    encontrado = re.search(r'(?:19|20)\d{2}', str(caminho))
    return int(encontrado.group(0)) if encontrado else None

def selecionar_arquivo_principal(caminhos, ano):
    if ano <= 2006:
        esperado = f'censoesc_{ano}.csv'
    elif ano == 2025:
        esperado = f'tabela_escola_{ano}.csv'
    else:
        esperado = f'microdados_ed_basica_{ano}.csv'

    for caminho in caminhos:
        if caminho.name.lower() == esperado:
            return caminho
    raise FileNotFoundError(f'Arquivo principal não encontrado para {ano}: {esperado}')

def identificar_csv(caminho):
    forcar_barra = bool(re.search(r'^(CENSOESC|EDUCPROF|INDIC|MEDPROF|EM\d+|ES\d+)', caminho.name, re.I))
    codificacoes = ['latin1', 'utf-8-sig'] if forcar_barra else ['utf-8-sig', 'latin1']

    for codificacao in codificacoes:
        try:
            if forcar_barra:
                colunas = pd.read_csv(caminho, sep='|', encoding=codificacao, encoding_errors='replace', nrows=0, engine='python').columns.tolist()
                return {'separador': '|', 'codificacao': codificacao, 'colunas': colunas}

            amostra = caminho.read_text(encoding='latin1', errors='replace')[:50_000]
            primeira_linha = amostra.splitlines()[0] if amostra else ''
            contagens = {sep: primeira_linha.count(sep) for sep in [';', '|', '\t', ',']}
            separador = max(contagens, key=contagens.get)
            if contagens[separador] == 0:
                separador = csv.Sniffer().sniff(amostra, delimiters=';,|\t,').delimiter

            colunas = pd.read_csv(caminho, sep=separador, encoding=codificacao, encoding_errors='replace', nrows=0, engine='python').columns.tolist()
            return {'separador': separador, 'codificacao': codificacao, 'colunas': colunas}
        except Exception:
            continue

    raise ValueError(f'Não foi possível ler o cabeçalho de {caminho}')

arquivos_csv_por_ano = {ano: [] for ano in ANOS}
for caminho in sorted(DIR_DADOS.rglob('*')):
    if caminho.suffix.lower() != '.csv':
        continue
    ano = extrair_ano(caminho)
    if ano in arquivos_csv_por_ano:
        arquivos_csv_por_ano[ano].append(caminho)

plano = []
for ano in ANOS:
    principal = selecionar_arquivo_principal(arquivos_csv_por_ano[ano], ano)
    config_csv = identificar_csv(principal)
    disponiveis = set(config_csv['colunas'])

    colunas_fonte = []
    for coluna_destino in COLUNAS_SAIDA:
        for apelido in ESPECIFICACOES_COLUNAS[coluna_destino]['apelidos']:
            if apelido in disponiveis and apelido not in colunas_fonte:
                colunas_fonte.append(apelido)

    assert len(colunas_fonte) <= MAX_COLUNAS_FONTE, (ano, len(colunas_fonte))

    plano.append({
        'ano': ano,
        'caminho': principal,
        'separador': config_csv['separador'],
        'codificacao': config_csv['codificacao'],
        'colunas': config_csv['colunas'],
        'colunas_fonte': colunas_fonte,
    })

pd.DataFrame([{
    'ano': item['ano'],
    'arquivo': item['caminho'].name,
    'separador': item['separador'],
    'codificacao': item['codificacao'],
    'colunas_originais': len(item['colunas']),
    'colunas_lidas': len(item['colunas_fonte']),
} for item in plano])


,ano,arquivo,separador,codificacao,colunas_originais,colunas_lidas
0,1995,CENSOESC_1995.CSV,|,latin1,479,13
1,1996,CENSOESC_1996.CSV,|,latin1,812,15
2,1997,CENSOESC_1997.CSV,|,latin1,367,24
3,1998,CENSOESC_1998.CSV,|,latin1,893,25
4,1999,CENSOESC_1999.CSV,|,latin1,1065,26
5,2000,CENSOESC_2000.CSV,|,latin1,872,25
6,2001,CENSOESC_2001.CSV,|,latin1,1244,27
7,2002,CENSOESC_2002.CSV,|,latin1,1307,27
8,2003,CENSOESC_2003.CSV,|,latin1,1853,27
9,2004,CENSOESC_2004.CSV,|,latin1,3260,29


### 4) Normalização

In [4]:
def limpar_serie(serie):
    return (
        serie.astype('string')
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
        .replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA, '<NA>': pd.NA})
    )

def primeira_serie_disponivel(lote, apelidos):
    selecionadas = [limpar_serie(lote[a]) for a in apelidos if a in lote.columns]
    if not selecionadas:
        return pd.Series(pd.NA, index=lote.index, dtype='string')
    saida = selecionadas[0]
    for serie in selecionadas[1:]:
        saida = saida.fillna(serie)
    return saida

def serie_inteira(lote, apelidos):
    return pd.to_numeric(primeira_serie_disponivel(lote, apelidos), errors='coerce').astype('Int64')

def indicador_a_partir_da_serie(serie):
    valor = limpar_serie(serie).str.lower()
    numero = pd.to_numeric(valor.str.replace(',', '.', regex=False), errors='coerce')

    saida = pd.Series(pd.NA, index=serie.index, dtype='Int8')
    saida[valor.isin({'1', 's', 'sim', 'y', 'yes', 'true', 'ativo'}) | (numero > 0)] = 1
    saida[valor.isin({'0', 'n', 'nao', 'não', 'no', 'false', 'inativo'}) | (numero == 0)] = 0
    return saida

def serie_indicadora(lote, apelidos):
    selecionadas = [indicador_a_partir_da_serie(lote[a]) for a in apelidos if a in lote.columns]
    if not selecionadas:
        return pd.Series(pd.NA, index=lote.index, dtype='Int8')
    saida = selecionadas[0]
    for serie in selecionadas[1:]:
        saida = saida.fillna(serie)
        saida = saida.mask((saida == 0) & (serie == 1), 1)
    return saida.astype('Int8')

def serie_dependencia(lote, apelidos):
    bruto = primeira_serie_disponivel(lote, apelidos)
    chave = bruto.str.lower()
    mapeado = chave.map({
        '1': 'Federal', 'federal': 'Federal',
        '2': 'Estadual', 'estadual': 'Estadual',
        '3': 'Municipal', 'municipal': 'Municipal',
        '4': 'Privada', 'particular': 'Privada', 'privada': 'Privada',
    })
    return mapeado.astype('string').fillna(bruto)

def serie_localizacao(lote, apelidos):
    bruto = primeira_serie_disponivel(lote, apelidos)
    chave = bruto.str.lower()
    mapeado = chave.map({'1': 'Urbana', 'urbana': 'Urbana', '2': 'Rural', 'rural': 'Rural'})
    return mapeado.astype('string').fillna(bruto)

def serie_situacao(lote, apelidos):
    bruto = primeira_serie_disponivel(lote, apelidos)
    chave = bruto.str.lower()
    mapeado = chave.map({
        '1': 'Ativa', 'ativo': 'Ativa', 'em atividade': 'Ativa',
        '2': 'Paralisada', 'paralisada': 'Paralisada',
        '3': 'Extinta', 'extinta': 'Extinta',
    })
    return mapeado.astype('string').fillna(bruto)

def serie_saida(lote, coluna_destino):
    info = ESPECIFICACOES_COLUNAS[coluna_destino]
    tipo = info['tipo']
    apelidos = info['apelidos']

    if tipo == 'inteiro':
        return serie_inteira(lote, apelidos)
    if tipo == 'indicador':
        return serie_indicadora(lote, apelidos)
    if tipo == 'categoria_dependencia':
        return serie_dependencia(lote, apelidos)
    if tipo == 'categoria_localizacao':
        return serie_localizacao(lote, apelidos)
    if tipo == 'categoria_situacao':
        return serie_situacao(lote, apelidos)

    return primeira_serie_disponivel(lote, apelidos)

def normalizar_lote(lote):
    saida = pd.DataFrame(index=lote.index)
    for coluna in COLUNAS_SAIDA:
        saida[coluna] = serie_saida(lote, coluna)
    return saida.loc[:, COLUNAS_SAIDA]


### 5) Tabelas auxiliares de 2025

In [5]:
def encontrar_tabela_2025(prefixo):
    for caminho in arquivos_csv_por_ano[2025]:
        if caminho.name.lower().startswith(prefixo.lower()):
            return caminho
    return None

def ler_tabela_auxiliar_2025(prefixo):
    caminho = encontrar_tabela_2025(prefixo)
    if caminho is None:
        return None

    config_csv = identificar_csv(caminho)
    disponiveis = set(config_csv['colunas'])

    desejadas = {'NU_ANO_CENSO', 'CO_ENTIDADE'}
    for coluna in COLUNAS_SAIDA:
        for apelido in ESPECIFICACOES_COLUNAS[coluna]['apelidos']:
            if apelido in disponiveis:
                desejadas.add(apelido)

    colunas_usadas = sorted(desejadas & disponiveis)
    if len(colunas_usadas) <= 2:
        return None

    df = pd.read_csv(
        caminho,
        sep=config_csv['separador'],
        encoding=config_csv['codificacao'],
        encoding_errors='replace',
        usecols=colunas_usadas,
        dtype='string',
    )

    if 'NU_ANO_CENSO' in df.columns:
        df['NU_ANO_CENSO'] = limpar_serie(df['NU_ANO_CENSO'])
    if 'CO_ENTIDADE' in df.columns:
        df['CO_ENTIDADE'] = limpar_serie(df['CO_ENTIDADE'])
    return df

auxiliares_2025 = []
for prefixo in ['Tabela_Matricula_2025', 'Tabela_Docente_2025', 'Tabela_Turma_2025']:
    df_auxiliar = ler_tabela_auxiliar_2025(prefixo)
    if df_auxiliar is not None:
        auxiliares_2025.append(df_auxiliar)

print(f'Tabelas auxiliares 2025 carregadas: {len(auxiliares_2025)}')


Tabelas auxiliares 2025 carregadas: 3


### 6) Escrita do CSV consolidado

In [6]:
def iterar_lotes_ano(item):
    leitor = pd.read_csv(
        item['caminho'],
        sep=item['separador'],
        encoding=item['codificacao'],
        encoding_errors='replace',
        usecols=item['colunas_fonte'],
        dtype='string',
        chunksize=TAMANHO_CHUNK,
        engine='python' if item['separador'] == '|' else 'c',
        on_bad_lines='warn',
    )

    for lote in leitor:
        if item['ano'] == 2025 and auxiliares_2025:
            if 'NU_ANO_CENSO' in lote.columns:
                lote['NU_ANO_CENSO'] = limpar_serie(lote['NU_ANO_CENSO'])
            if 'CO_ENTIDADE' in lote.columns:
                lote['CO_ENTIDADE'] = limpar_serie(lote['CO_ENTIDADE'])

            for df_auxiliar in auxiliares_2025:
                lote = lote.merge(df_auxiliar, on=['NU_ANO_CENSO', 'CO_ENTIDADE'], how='left', suffixes=('', '_aux'))

        yield normalizar_lote(lote)

if CAMINHO_SAIDA.exists():
    CAMINHO_SAIDA.unlink()

linhas_por_ano = {}
cabecalho_escrito = False

for item in plano:
    linhas_ano = 0
    for df_lote in iterar_lotes_ano(item):
        df_lote.to_csv(
            CAMINHO_SAIDA,
            mode='a',
            index=False,
            header=not cabecalho_escrito,
            encoding='utf-8',
        )
        cabecalho_escrito = True
        linhas_ano += len(df_lote)

    linhas_por_ano[item['ano']] = linhas_ano
    print(f"{item['ano']}: {linhas_ano:,} linhas")

print(f'CSV consolidado gerado: {CAMINHO_SAIDA}')


1995: 243,637 linhas
1996: 276,731 linhas
1997: 273,951 linhas
1998: 267,532 linhas
1999: 266,645 linhas
2000: 261,988 linhas
2001: 264,735 linhas
2002: 256,986 linhas
2003: 253,405 linhas
2004: 248,257 linhas
2005: 248,103 linhas
2006: 241,817 linhas
2007: 198,507 linhas
2008: 205,699 linhas
2009: 203,455 linhas
2010: 200,876 linhas
2011: 242,147 linhas
2012: 242,136 linhas
2013: 242,680 linhas
2014: 242,929 linhas
2015: 237,879 linhas
2016: 237,506 linhas
2017: 236,481 linhas
2018: 236,460 linhas
2019: 228,521 linhas
2020: 224,229 linhas
2021: 221,140 linhas
2022: 224,649 linhas
2023: 217,625 linhas
2024: 215,545 linhas
2025: 214,192 linhas
CSV consolidado gerado: /home/gabriel/ciencias-computacao/programacao-para-analise-de-dados/trabalho/censo-escolar.csv


### 7) Validação rápida

In [7]:
total_linhas = 0
ano_min = None
ano_max = None
amostra = None
colunas = pd.read_csv(CAMINHO_SAIDA, nrows=0).columns.tolist()

for lote in pd.read_csv(CAMINHO_SAIDA, usecols=['ano_censo'], chunksize=250_000):
    anos = pd.to_numeric(lote['ano_censo'], errors='coerce')
    cur_min = anos.min(skipna=True)
    cur_max = anos.max(skipna=True)
    ano_min = cur_min if ano_min is None else min(ano_min, cur_min)
    ano_max = cur_max if ano_max is None else max(ano_max, cur_max)
    total_linhas += len(lote)

amostra = pd.read_csv(CAMINHO_SAIDA, nrows=5)

print(f'Linhas: {total_linhas:,}')
print(f'Colunas: {len(colunas)}')
print(f'Tamanho: {CAMINHO_SAIDA.stat().st_size / 1024**2:.1f} MB')
print(f'Anos: {(ano_min, ano_max)}')

amostra


Linhas: 7,376,443
Colunas: 42
Tamanho: 1074.8 MB
Anos: (np.int64(1995), np.int64(2025))


,ano_censo,id_escola,co_municipio,no_municipio,sg_uf,co_uf,no_uf,no_regiao,co_regiao,no_entidade,...,in_energia_inexistente,qt_mat_bas,qt_mat_fund,qt_mat_med,qt_doc_bas,qt_doc_fund,qt_doc_med,qt_tur_bas,qt_tur_fund,qt_tur_med
0,1995,27,31070300620005,BELO HORIZONTE,MG,NaN,Minas Gerais,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1995,35,31070300620005,BELO HORIZONTE,MG,NaN,Minas Gerais,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1995,43,31070300620005,BELO HORIZONTE,MG,NaN,Minas Gerais,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1995,51,31070300620005,BELO HORIZONTE,MG,NaN,Minas Gerais,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1995,60,31070300620005,BELO HORIZONTE,MG,NaN,Minas Gerais,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
